<a href="https://colab.research.google.com/github/josenomberto/UTEC-CDIAV3-MCD8016/blob/main/Session3_0_EfficientNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class SqueezeExcitation(nn.Module):
    def __init__(self, in_chan, se_ratio=0.25):
        super(SqueezeExcitation, self).__init__()
        n_hidden = int(in_chan * se_ratio)

        self.se_block = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)), # Promedio en las dimensiones espaciales. Similar a la ResNet
            # [n_batch,n_chan,1,1]

            # Funciona como una MLP
            nn.Conv2d(in_chan,n_hidden,kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(n_hidden,in_chan,kernel_size=1),
            nn.Sigmoid()
        )

    # x: [n_batch,n_chan,h,w]
    def forward(self, x):
        # Vector de atencion de canales
        # attn: [n_batch,n_chan,1,1]
        attn = self.se_block(x)
        return x * attn

In [ ]:
class MBConv(nn.Module):
    def __init__(self, in_chan, out_chan, stride, expand_ratio, se_ratio=0.25):
        super(MBConv, self).__init__()
        n_hidden = in_chan * expand_ratio
        self.use_residual = (in_chan == out_chan and stride == 1)

        # Etapa de expansión (solo se usa si expand_ratio > 1)
        layers = []
        if expand_ratio > 1:
            layers.append(nn.Conv2d(in_chan, n_hidden, kernel_size=1, bias=False))
            layers.append(nn.BatchNorm2d(n_hidden))
            layers.append(nn.ReLU6(inplace=True))

        # Convolución depthwise
        layers.append(nn.Conv2d(n_hidden, n_hidden, kernel_size=3, stride=stride, padding=1, groups=n_hidden, bias=False))
        layers.append(nn.BatchNorm2d(n_hidden))
        layers.append(nn.ReLU6(inplace=True))

        # Squeeze-and-Excitation block
        layers.append(SqueezeExcitation(n_hidden, se_ratio))

        # Proyección
        layers.append(nn.Conv2d(n_hidden, out_chan, kernel_size=1, bias=False))
        layers.append(nn.BatchNorm2d(out_chan))

        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        identity = x

        x = self.conv(x)

        if self.use_residual:
            x += identity
        return x

In [ ]:
class EfficientNetB0(nn.Module):
    def __init__(self, n_classes=1000):
        super(EfficientNetB0, self).__init__()

        # Capa de entrada
        self.block0 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True)
        )

        # Bloque 1
        self.block1 = MBConv(32, 16, stride=1, expand_ratio=1)

        # Bloque 2
        self.block2 = nn.Sequential(
            MBConv(16, 24, stride=2, expand_ratio=6),
            MBConv(24, 24, stride=1, expand_ratio=6)
        )

        # Bloque 3
        self.block3 = nn.Sequential(
            MBConv(24, 40, stride=2, expand_ratio=6),
            MBConv(40, 40, stride=1, expand_ratio=6)
        )

        # Bloque 4
        self.block4 = nn.Sequential(
            MBConv(40, 80, stride=2, expand_ratio=6),
            MBConv(80, 80, stride=1, expand_ratio=6),
            MBConv(80, 80, stride=1, expand_ratio=6)
        )

        # Bloque 5
        self.block5 = nn.Sequential(
            MBConv(80, 112, stride=1, expand_ratio=6),
            MBConv(112, 112, stride=1, expand_ratio=6),
            MBConv(112, 112, stride=1, expand_ratio=6)
        )

        # Bloque 6
        self.block6 = nn.Sequential(
            MBConv(112, 192, stride=2, expand_ratio=6),
            MBConv(192, 192, stride=1, expand_ratio=6),
            MBConv(192, 192, stride=1, expand_ratio=6),
            MBConv(192, 192, stride=1, expand_ratio=6)
        )

        # Bloque 7
        self.block7 = MBConv(192, 320, stride=1, expand_ratio=6)

        # Bloque 8
        self.block8 = nn.Sequential(
            nn.Conv2d(320, 1280, kernel_size=1, bias=False),
            nn.BatchNorm2d(1280),
            nn.ReLU6(inplace=True)
        )

        # Capa de clasificación
        self.classifier = nn.Linear(1280, n_classes)

    def forward(self, x):
        x = self.block0(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.block6(x)
        x = self.block7(x)
        x = self.block8(x)

        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x